Cell 1: Import thư viện & Cấu hình đường dẫn cây thư mục

In [2]:
import os
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split

DATA_DIR = os.path.join("..", "..", "data", "processed")

MODEL_DIR = os.path.join("..", "saved_models")
os.makedirs(MODEL_DIR, exist_ok=True)

ratings_path = os.path.join(DATA_DIR, "ratings_cleaned.csv")
books_path = os.path.join(DATA_DIR, "books_cleaned.csv")

df_ratings = pd.read_csv(ratings_path, low_memory=False).dropna(how="all")
df_books = pd.read_csv(books_path, low_memory=False).dropna(how="all")

print(f"✅ Ratings Data ({len(df_ratings):,} dòng): {ratings_path}")
print(f"✅ Books Data ({len(df_books):,} dòng): {books_path}")

✅ Ratings Data (383,842 dòng): ..\..\data\processed\ratings_cleaned.csv
✅ Books Data (271,360 dòng): ..\..\data\processed\books_cleaned.csv


Cell 2: Chia tập dữ liệu Ratings 60/20/20

In [3]:
train_ratings, temp_ratings = train_test_split(df_ratings, test_size=0.40, random_state=42)
val_ratings, test_ratings = train_test_split(temp_ratings, test_size=0.50, random_state=42)
print(f"📊 Ratings Train (60%): {len(train_ratings)} dòng")
print(f"📊 Ratings Validation (20%): {len(val_ratings)} dòng")
print(f"📊 Ratings Test (20%): {len(test_ratings)} dòng")

📊 Ratings Train (60%): 230305 dòng
📊 Ratings Validation (20%): 76768 dòng
📊 Ratings Test (20%): 76769 dòng


Cell 3: Tính toán chỉ số Popularity trên tập Train

In [4]:
id_col = 'book_id' if 'book_id' in df_books.columns else ('ISBN' if 'ISBN' in df_books.columns else df_books.columns[0])
title_col = 'title' if 'title' in df_books.columns else ('Book-Title' if 'Book-Title' in df_books.columns else df_books.columns[1])
author_col = 'author' if 'author' in df_books.columns else ('Book-Author' if 'Book-Author' in df_books.columns else df_books.columns[2])
r_id_col = id_col if id_col in train_ratings.columns else ('ISBN' if 'ISBN' in train_ratings.columns else train_ratings.columns[1])
r_rating_col = 'rating' if 'rating' in train_ratings.columns else ('Book-Rating' if 'Book-Rating' in train_ratings.columns else train_ratings.columns[2])
train_ratings[r_rating_col] = pd.to_numeric(train_ratings[r_rating_col], errors='coerce')
stats = train_ratings.groupby(r_id_col).agg(
    rating_count=(r_rating_col, 'count'),
    rating_mean=(r_rating_col, 'mean')
).reset_index()
popular_df = stats[stats['rating_count'] >= 10].sort_values(by=['rating_mean', 'rating_count'], ascending=False)
popular_books_df = popular_df.merge(df_books[[id_col, title_col, author_col]], left_on=r_id_col, right_on=id_col, how='inner')

Cell 4: Hàm gợi ý Top sách cho User mới (Cold Start)

In [5]:
def get_popular_books(top_k=5):
    top_list = popular_books_df.head(top_k)
    result = [{
        "book_id": str(row[id_col]),
        "title": str(row[title_col]),
        "score": round(float(row['rating_mean']), 2)
    } for _, row in top_list.iterrows()]
    return {"recommend_type": "Popularity (Cold Start)", "popular_books": result}
print("🔥 Top 3 sách phổ biến nhất (Cold Start):")
print(get_popular_books(top_k=3))

🔥 Top 3 sách phổ biến nhất (Cold Start):
{'recommend_type': 'Popularity (Cold Start)', 'popular_books': [{'book_id': '0439425220', 'title': 'Harry Potter and the Chamber of Secrets Postcard Book', 'score': 9.92}, {'book_id': '0394800389', 'title': 'Fox in Socks (I Can Read It All by Myself Beginner Books)', 'score': 9.9}, {'book_id': '0920668372', 'title': 'Love You Forever', 'score': 9.8}]}


Cell 5: Đóng gói và lưu Model vào thư mục

In [6]:
export_path = os.path.join(MODEL_DIR, "popularity_model.pkl")
model_data_pop = {
    'popular_books_df': popular_books_df[[id_col, title_col, author_col, 'rating_count', 'rating_mean']],
    'id_col': id_col,
    'title_col': title_col
}
joblib.dump(model_data_pop, export_path, compress=3)
print(f"🎉 THÀNH CÔNG! Đã xuất file model chuẩn tại: {export_path}")

🎉 THÀNH CÔNG! Đã xuất file model chuẩn tại: ..\saved_models\popularity_model.pkl
